# lineup — model stages on a free cloud GPU

This notebook runs the GPU stages of the benchmark on a free Colab or Kaggle T4 (16 GB). It clones the repository, builds test cases on CPU, and loads Qwen2.5-7B-Instruct in 4-bit to run generation and teacher-forced scoring.

Before running, set the runtime to a GPU: **Runtime → Change runtime type → T4 GPU** (on Kaggle, enable the GPU accelerator and Internet in the sidebar).

In [ ]:
!git clone https://github.com/santoshcheethiralame-dot/LINEUP
%cd LINEUP
!pip install -q -e .
!pip install -q bitsandbytes

## Build test cases (CPU)

Scenario construction is model-free and deterministic, so the same seed reproduces the same benchmark on any machine.

In [ ]:
!python scripts/build_scenarios.py --limit 100

## Load Qwen2.5-7B in 4-bit

The first load downloads about 5 GB of 4-bit weights.

In [ ]:
from lineup.backends import Message, TransformersModel
from lineup.config import DEFAULT_MODEL, set_seed

set_seed()
model = TransformersModel(DEFAULT_MODEL, load_in_4bit=True, max_new_tokens=64)

context = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
    "It is named after the engineer Gustave Eiffel, whose company built it between 1887 and 1889."
)
messages = [
    Message("system", "Answer the question using only the provided context. Be concise."),
    Message("user", f"Context:\n{context}\n\nQuestion: Who is the Eiffel Tower named after?"),
]

generation = model.generate(messages)
print("answer:", generation.text.strip())

scoring = model.score(messages, generation.text)
print("total logprob:", round(scoring.total_logprob, 3))

## Stage 3 — generation and correctness

Build a batch of cases, run the model on each, and count how often it answered wrongly — and of those, how often it echoed the value the misleading chunk was built to induce. Reuses the 4-bit `model` loaded above as both the answerer and the correctness judge.

In [ ]:
from lineup.data.hotpotqa import load_examples
from lineup.data.substitution import build_answer_pool
from lineup.data.scenario import ScenarioBuilder
from lineup.data.misleading import substitution_check
from lineup.correctness import LLMJudge
from lineup.generation import generate_and_judge

examples = list(load_examples("validation", limit=50))
pool = build_answer_pool(examples)
builder = ScenarioBuilder(answer_pool=pool, seed=0)
judge = LLMJudge(model)

correct = wrong = fooled = 0
for example in examples:
    if substitution_check(example):
        continue
    scenario = builder.build(example)
    if scenario is None:
        continue
    result = generate_and_judge(model, scenario, llm_judge=judge)
    correct += result.is_correct
    if not result.is_correct:
        wrong += 1
        fooled += result.matched_intended_wrong

print(f"correct={correct} wrong={wrong} echoed-planted-value={fooled}")

## Notes

- 4-bit (nf4) keeps the 7B model within a 16 GB T4; fp16 compute is used because the T4 has no native bfloat16.
- Run **all** model stages on one machine and one pinned model revision — greedy decoding is deterministic per machine, but log-probabilities drift across GPUs and precisions, so the labels must come from a single box.
- The scenarios themselves are machine-independent, so they can be rebuilt anywhere from the same seed.